In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
## Data From: https://www.dubaipulse.gov.ae/data/dld-transactions/dld_transactions-open-api

In [2]:
def load_and_clean_data(filepath: str):  
    ## Load Data -------------------------------------------
    use_cols = [
    "trans_group_en",
    "procedure_name_en",
    "instance_date",
    "property_type_en",
    "property_sub_type_en",
    "property_usage_en",
    "reg_type_en",
    "area_name_en",
    "building_name_en",
    "project_name_en",
    "master_project_en",
    "nearest_landmark_en",
    "nearest_metro_en",
    "nearest_mall_en",
    "rooms_en",
    "has_parking",
    "procedure_area",
    "actual_worth",
    "meter_sale_price",
    "rent_value",
    "meter_rent_price",
    "no_of_parties_role_1",
    "no_of_parties_role_2",
    "no_of_parties_role_3"
    ]

    pd.set_option("display.max_columns", None)
    transactions = pd.read_csv(filepath, usecols=use_cols)


    ## Convert types -------------------------------------------
    cat_cols = [
        "trans_group_en",
        "procedure_name_en",
        "property_type_en",
        "property_sub_type_en",
        "property_usage_en",
        "reg_type_en",
        "area_name_en",
        "building_name_en",
        "project_name_en",
        "master_project_en",
        "nearest_landmark_en",
        "nearest_metro_en",
        "nearest_mall_en",
        "rooms_en"
    ]

    transactions["instance_date"] = pd.to_datetime( transactions["instance_date"], dayfirst=True, errors="coerce")

    for col in cat_cols:
        transactions[col] = transactions[col].str.strip().str.lower().astype("category")

    transactions["has_parking"] = transactions["has_parking"].astype("bool")
    transactions["procedure_area"] = pd.to_numeric(transactions["procedure_area"], errors="coerce").astype("float32")
    transactions["actual_worth"] = pd.to_numeric(transactions["actual_worth"], errors="coerce").astype("float64")
    transactions["meter_sale_price"] = pd.to_numeric(transactions["meter_sale_price"], errors="coerce").astype("float32")
    transactions["rent_value"] = pd.to_numeric(transactions["rent_value"], errors="coerce").astype("float32")
    transactions["meter_rent_price"] = pd.to_numeric(transactions["meter_rent_price"], errors="coerce").astype("float32")

    transactions["no_of_parties_role_1"] = pd.to_numeric(transactions["no_of_parties_role_1"], errors="coerce").astype("Int16")
    transactions["no_of_parties_role_2"] = pd.to_numeric(transactions["no_of_parties_role_2"], errors="coerce").astype("Int16")
    transactions["no_of_parties_role_3"] = pd.to_numeric(transactions["no_of_parties_role_3"], errors="coerce").astype("Int16")

    transactions = transactions.dropna(subset=["instance_date", "actual_worth", "procedure_area"], how="any").reset_index(drop=True)

    return transactions


In [3]:
df = load_and_clean_data("data/Transactions.csv")
df.head()

,trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,building_name_en,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3
0,gifts,grant,2006-10-16,villa,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,3162.419922,12000000.0,3794.560059,NaN,NaN,3,1,0
1,gifts,grant,2019-11-13,land,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,209.089996,916659.0,4384.040039,NaN,NaN,2,4,0
2,mortgages,mortgage registration,1999-03-22,land,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,1062.719971,1200000.0,1129.180054,NaN,NaN,1,1,0
3,mortgages,mortgage registration,2001-08-20,building,NaN,residential / commercial,existing properties,oud metha,NaN,NaN,NaN,burj khalifa,oud metha metro station,dubai mall,NaN,False,1337.800049,4519342.0,3378.189941,NaN,NaN,1,1,0
4,mortgages,mortgage registration,2020-11-30,building,NaN,residential,existing properties,al bada,NaN,NaN,NaN,burj khalifa,trade centre metro station,dubai mall,NaN,False,278.709991,2500000.0,8969.900391,NaN,NaN,1,1,0


In [4]:
def time_filter(df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    mask = (df["instance_date"] >= start_date) & (df["instance_date"] <= end_date)
    return df[mask].reset_index(drop=True)

In [5]:
df["rooms_en"].cat.categories

Index(['1 b/r', '10 b/r', '2 b/r', '3 b/r', '4 b/r', '5 b/r', '6 b/r', '7 b/r',
       '8 b/r', '9 b/r', 'gym', 'office', 'penthouse', 'shop', 'single room',
       'store', 'studio'],
      dtype='object')

In [44]:

def map_advertised_area(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    advertised_area = pd.read_csv("data/dubai_area_alias_mapping.csv")
    advertised_area["official_area"] = advertised_area["official_area"].str.strip().str.lower().astype("category")
    advertised_area["advertised_area"] = advertised_area["advertised_area"].str.strip().str.lower().astype("category")

    area_map = advertised_area.groupby("official_area")["advertised_area"].agg(lambda x: " | ".join(sorted(set(x))))
    df["advertised_area"] = df["area_name_en"].map(area_map).astype("category")
    return df

df = map_advertised_area(df)
df.head()

C:\Users\rayan\AppData\Local\Temp\ipykernel_22648\4213932629.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  area_map = advertised_area.groupby("official_area")["advertised_area"].agg(lambda x: " | ".join(sorted(set(x))))


,trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,building_name_en,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3,advertised_area
0,gifts,grant,2006-10-16,villa,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,3162.419922,12000000.0,3794.560059,NaN,NaN,3,1,0,NaN
1,gifts,grant,2019-11-13,land,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,209.089996,916659.0,4384.040039,NaN,NaN,2,4,0,NaN
2,mortgages,mortgage registration,1999-03-22,land,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,1062.719971,1200000.0,1129.180054,NaN,NaN,1,1,0,NaN
3,mortgages,mortgage registration,2001-08-20,building,NaN,residential / commercial,existing properties,oud metha,NaN,NaN,NaN,burj khalifa,oud metha metro station,dubai mall,NaN,False,1337.800049,4519342.0,3378.189941,NaN,NaN,1,1,0,NaN
4,mortgages,mortgage registration,2020-11-30,building,NaN,residential,existing properties,al bada,NaN,NaN,NaN,burj khalifa,trade centre metro station,dubai mall,NaN,False,278.709991,2500000.0,8969.900391,NaN,NaN,1,1,0,NaN


In [50]:
df.loc[df["trans_group_en"] == "gifts"]

,trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,building_name_en,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3,advertised_area
0,gifts,grant,2006-10-16,villa,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,3162.419922,12000000.0,3794.560059,NaN,NaN,3,1,0,NaN
1,gifts,grant,2019-11-13,land,NaN,residential,existing properties,mankhool,NaN,NaN,NaN,burj khalifa,adcb metro station,dubai mall,NaN,False,209.089996,916659.0,4384.040039,NaN,NaN,2,4,0,NaN
15,gifts,grant,2010-03-29,building,NaN,residential,existing properties,al raffa,NaN,NaN,NaN,burj khalifa,al ghubaiba metro station,dubai mall,NaN,False,393.440002,5395711.0,13714.190430,NaN,NaN,1,1,0,NaN
19,gifts,grant,2025-06-12,building,NaN,residential,existing properties,al satwa,NaN,NaN,NaN,burj khalifa,trade centre metro station,dubai mall,NaN,False,105.290001,2000001.0,18995.169922,NaN,NaN,1,2,0,al satwa | satwa
68,gifts,grant,2025-07-03,villa,NaN,residential,existing properties,mirdif,NaN,NaN,NaN,dubai international airport,rashidiya metro station,city centre mirdif,NaN,False,6967.729980,42500001.0,6099.549805,NaN,NaN,1,1,0,mirdif
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1636829,gifts,grant,2025-08-18,unit,flat,residential,existing properties,al barshaa south third,elz residence,elz residence,arjan,motor city,sharaf dg metro station,mall of the emirates,studio,True,36.480000,483715.0,13259.740234,NaN,NaN,1,1,0,al barshaa south third
1636841,gifts,grant,2025-09-11,unit,flat,residential,existing properties,al barsha south fourth,living garden ii,living garden ii,jumeirah village circle,sports city swimming academy,nakheel metro station,marina mall,studio,True,51.810001,592251.0,11431.219727,NaN,NaN,1,1,0,jumeirah village circle
1636870,gifts,grant pre-registration,2023-11-27,unit,flat,residential,off-plan properties,hadaeq sheikh mohammed bin rashid,mag 960,mag eye phase 1,hadaeq sheikh mohammed bin rashid - disrict 7,NaN,NaN,NaN,studio,True,38.250000,618616.0,16172.969727,NaN,NaN,1,1,0,dubai hills
1636976,gifts,grant,2025-10-29,land,NaN,residential,existing properties,madinat dubai almelaheyah,NaN,31 above by beyond,dubai maritime city,NaN,NaN,NaN,NaN,False,8376.870117,136157645.0,16254.000000,NaN,NaN,1,1,0,madinat dubai al melaheyah
